```
# Lab type:  debug
# Course:    NL301 Natural Language Processing with Python
# Lesson:    02 — Tokenisation and Normalisation
# Task:      Find and fix three silent bugs in a text preprocessing pipeline.
```

## Setup and reference code

Run this cell first. It installs dependencies and defines a **correct** reference implementation you can compare against.

In [ ]:
!pip install spacy nltk scikit-learn --quiet
import spacy, nltk, re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

nltk.download('stopwords', quiet=True)
nlp = spacy.load('en_core_web_sm')

# Reference — correct preprocessing (DO NOT change this cell)
SAFE_STOPWORDS = set(stopwords.words('english')) - {'not', 'no', 'nor', 'never', 'without'}

def preprocess_correct(text):
    doc = nlp(text)                          # NER before lowercasing
    tokens = [t.text for t in doc if not t.is_punct and not t.is_space]
    tokens = [t.lower() for t in tokens if t.lower() not in SAFE_STOPWORDS]
    return ' '.join(tokens)

sample = "Apple CEO Tim Cook said the product is not bad."
print("Correct output:", preprocess_correct(sample))


---
## Bug 1: Lowercasing before NER

**What you'll see:** `nlp()` produces no `PERSON` or `ORG` entities for `"Apple CEO Tim Cook"`.

The cell below lowercases the string **before** passing it to spaCy. Since spaCy's NER model was trained on mixed-case text, lowercasing destroys the cues it relies on.

In [ ]:
# BUG: text is lowercased before NER
def preprocess_buggy_1(text):
    text = text.lower()                      # ← Bug: destroys capitalisation
    doc = nlp(text)
    print("Entities found:", [(ent.text, ent.label_) for ent in doc.ents])
    tokens = [t.text for t in doc if not t.is_punct and not t.is_space]
    tokens = [t for t in tokens if t not in SAFE_STOPWORDS]
    return ' '.join(tokens)

print(preprocess_buggy_1("Apple CEO Tim Cook said the product is not bad."))


**Explain the bug** — what output did you observe? Why does lowercasing before `nlp()` cause NER to fail?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**What the bug does:** `text.lower()` is called before `nlp(text)`, so spaCy's NER model receives all-lowercase input. NER was trained on mixed-case text and relies on capitalisation as a strong signal — "Apple" and "Tim Cook" are recognisable as ORG/PERSON precisely because they are capitalised. Lowercased, they are indistinguishable from common nouns and NER produces no entities.

**Correct approach:** Call `nlp(text)` first to extract entities from the original-cased string, then lowercase individual tokens during the filter step: `t.lower() for t in tokens if t.lower() not in SAFE_STOPWORDS`.

</details>

In [ ]:
# FIX 1: run NER first, lowercase after
def preprocess_fixed_1(text):
    doc = nlp(text)                          # NER on original-cased text
    print("Entities found:", [(ent.text, ent.label_) for ent in doc.ents])
    tokens = [t.text for t in doc if not t.is_punct and not t.is_space]
    tokens = [t.lower() for t in tokens if t.lower() not in SAFE_STOPWORDS]
    return ' '.join(tokens)

print(preprocess_fixed_1("Apple CEO Tim Cook said the product is not bad."))


---
## Bug 2: Stopword removal destroys negation

NLTK's default stopword list includes `"not"`, `"no"`, `"nor"`. Removing them before sentiment analysis silently flips meaning: `"not bad"` becomes `"bad"`.

In [ ]:
# BUG: default NLTK stopwords include negation words
nltk_stopwords = set(stopwords.words('english'))
print("'not' in default stopwords:", 'not' in nltk_stopwords)
print("'no'  in default stopwords:", 'no'  in nltk_stopwords)

def preprocess_buggy_2(text):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in nltk_stopwords]   # ← Bug
    return ' '.join(tokens)

reviews = ["not bad at all", "no problems with delivery", "this is never acceptable"]
for r in reviews:
    print(f"  '{r}' → '{preprocess_buggy_2(r)}'")


**Explain the bug** — what happened to the negation words? Why does this matter for sentiment analysis?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**What the bug does:** NLTK's default English stopword list includes `"not"`, `"no"`, `"nor"`, and `"never"`. Removing them silently inverts meaning: `"not bad"` → `"bad"`, `"no problems"` → `"problems"`, `"never acceptable"` → `"acceptable"`. A sentiment classifier trained or evaluated on this preprocessing will see flipped polarity for any negated phrase.

**Correct approach:** Subtract negation words from the stopword set before filtering: `set(stopwords.words('english')) - {'not', 'no', 'nor', 'never', 'without'}`. This preserves the tokens that carry the most sentiment-critical signal.

</details>

In [ ]:
# FIX 2: exclude negation words from the stopword list
SAFE_STOPWORDS_FIX = set(stopwords.words('english')) - {'not', 'no', 'nor', 'never', 'without'}

def preprocess_fixed_2(text):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in SAFE_STOPWORDS_FIX]
    return ' '.join(tokens)

for r in reviews:
    print(f"  '{r}' → '{preprocess_fixed_2(r)}'")


---
## Bug 3: Training / inference mismatch

The training code calls `preprocess()` and then `TfidfVectorizer.fit_transform()`. The inference code calls `vectorizer.transform([raw_text])` — skipping preprocessing. New text is vectorised differently from training text, silently degrading accuracy.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Tiny example corpus
texts  = ["great product", "not good at all", "love it", "terrible service",
          "works perfectly", "do not buy", "highly recommend", "awful quality"]
labels = [1, 0, 1, 0, 1, 0, 1, 0]

# BUG: fit_transform on raw text during training …
vectorizer_bug = TfidfVectorizer()
X = vectorizer_bug.fit_transform([preprocess_correct(t) for t in texts])
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.25, random_state=42)
clf_bug = LogisticRegression().fit(X_train, y_train)

# … but transform raw text during inference (skips preprocess)
new_text = "not bad at all"
pred = clf_bug.predict(vectorizer_bug.transform([new_text]))   # ← Bug: raw text
print("Buggy prediction:", pred)


**Explain the bug** — what is different between how training data and inference data enter the vectorizer?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**What the bug does:** During training, each text goes through `preprocess_correct()` before `fit_transform`, so the vectorizer learns a vocabulary of preprocessed tokens. During inference, `vectorizer.transform([raw_text])` is called on the raw string — no preprocessing applied. The same phrase (e.g., `"not bad"`) is represented differently at train vs inference time, silently degrading accuracy.

**Correct approach:** Wrap the preprocessor, vectorizer, and classifier in a sklearn `Pipeline`. `Pipeline.predict()` always applies every step in sequence, guaranteeing identical transformations at fit time and inference time.

</details>

In [ ]:
# FIX 3: wrap preprocessing + vectorizer + classifier in a sklearn Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

class TextPreprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X): return [preprocess_correct(t) for t in X]

pipe = Pipeline([
    ('prep', TextPreprocessor()),
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression()),
])
pipe.fit(texts, labels)
print("Fixed prediction:", pipe.predict(["not bad at all"]))
print("Training accuracy:", pipe.score(texts, labels))


---
## Summary

Fill in your answers:

1. Bug 1 fix: ___
2. Bug 2 fix: ___
3. Bug 3 fix: ___

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Bug 1:** Run `nlp()` before lowercasing so NER has access to the original capitalisation.
2. **Bug 2:** Exclude negation words (`not`, `no`, `nor`, `never`) from the NLTK stopword set.
3. **Bug 3:** Wrap preprocessing + vectorizer + classifier in a sklearn `Pipeline` to enforce consistent transformations at both train and inference time.

</details>